In [1]:
import geopandas as gpd
import libpysal
from esda.moran import Moran_Local
import numpy as np
from libpysal.weights import Queen
import pandas as pd
import rasterio
from pyproj import Geod
import rasterio
import geopandas as gpd
import numpy as np

c:\Users\Ducnghia\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# Đọc raster dự đoán từ file prediction.tif
pred_raster = rasterio.open(r'output/prediction.tif').read(1)

# Đọc raster chứa ID vùng từ file mastergrid_census2019.tif
fine_id_raster = rasterio.open(r'data_new/mastergrid_census2019.tif').read(1)

# Đọc shapefile các vùng hành chính từ file Census2019_All_of_Vietnam.shp
shapefile_path = r'data_new/admin/Census2019_All_of_Vietnam.shp'
gdf = gpd.read_file(shapefile_path)

In [13]:
mask = []
pred_pop = []
count = 0
for idx, row in gdf.iterrows():
    id = row['OBJECTID']
    total_pop = row['Total_popu']
    count += 1
    mask = fine_id_raster == id
    pred_sum = np.nansum(pred_raster[mask])
    pred_pop.append(pred_sum)
    print(f"{count}: OBJECTID={id}, Total_popu={total_pop}, pred_pop={pred_sum}")

1: OBJECTID=1, Total_popu=9574.0, pred_pop=6525.7177734375
2: OBJECTID=2, Total_popu=16538.0, pred_pop=13106.0498046875
3: OBJECTID=3, Total_popu=11625.0, pred_pop=7589.833984375
4: OBJECTID=4, Total_popu=6815.0, pred_pop=5098.92138671875
5: OBJECTID=5, Total_popu=9302.0, pred_pop=5819.125
6: OBJECTID=6, Total_popu=10605.0, pred_pop=6303.111328125
7: OBJECTID=7, Total_popu=9629.0, pred_pop=7572.29833984375
8: OBJECTID=8, Total_popu=14819.0, pred_pop=12266.9228515625
9: OBJECTID=9, Total_popu=8711.0, pred_pop=6693.75341796875
10: OBJECTID=10, Total_popu=11814.0, pred_pop=8723.591796875
11: OBJECTID=11, Total_popu=6744.0, pred_pop=4902.3095703125
12: OBJECTID=12, Total_popu=9989.0, pred_pop=7222.38427734375
13: OBJECTID=13, Total_popu=10078.0, pred_pop=9129.330078125
14: OBJECTID=14, Total_popu=10907.0, pred_pop=8614.6376953125
15: OBJECTID=15, Total_popu=17411.0, pred_pop=15286.1337890625
16: OBJECTID=16, Total_popu=13232.0, pred_pop=12850.34765625
17: OBJECTID=17, Total_popu=11064.0, p

In [ ]:
gdf['pred_pop'] = pred_pop
gdf["residual"] = gdf['pred_pop'] - gdf['Total_popu']
gdf[['Total_popu', 'pred_pop', "residual"]]

,Total_popu,pred_pop,residual
0,9574.0,6525.717773,-3048.282227
1,16538.0,13106.049805,-3431.950195
2,11625.0,7589.833984,-4035.166016
3,6815.0,5098.921387,-1716.078613
4,9302.0,5819.125000,-3482.875000
...,...,...,...
11156,3295.0,1692.029419,-1602.970581
11157,3761.0,3185.074463,-575.925537
11158,11979.0,9719.194336,-2259.805664
11159,3761.0,2766.287842,-994.712158


In [25]:
# Xuất kết quả pred_pop và residual ra file CSV riêng
output_csv = 'output/pred_pop_residual.csv'
gdf[['Total_popu', 'pred_pop', 'residual']].to_csv(output_csv, index=False)
print(f"Đã lưu kết quả ra {output_csv}")

Đã lưu kết quả ra output/pred_pop_residual.csv


In [18]:
w = libpysal.weights.KNN.from_dataframe(gdf, k=10)
lisa = Moran_Local(gdf["residual"], w)
labels = np.array(["", "HH", "LH", "LL", "HL"])
gdf["local_I"] = lisa.Is
gdf["p_value"] = lisa.p_sim
gdf["quadrant"] = labels[lisa.q]
gdf["z_score"] = lisa.z_sim

In [21]:
# Tính Moran toàn cục (Global Moran's I) cho residual
from esda.moran import Moran

global_moran = Moran(gdf["residual"], w)
print("Global Moran's I:", global_moran.I)
print("p-value:", global_moran.p_sim)
print("z-score:", global_moran.z_sim)


Global Moran's I: 0.2409298438076362
p-value: 0.001
z-score: 60.50671493626592


In [22]:
geod = Geod(ellps="WGS84")

def area_ha(geom):
    area, _ = geod.geometry_area_perimeter(geom)
    return abs(area) / 1e4  # m² → hectare

gdf["area_ha"] = gdf.geometry.apply(area_ha)
gdf['pop_density'] = gdf['Total_popu'] / gdf["area_ha"]

In [24]:
# Lưu kết quả ra shapefile trong thư mục output
output_path = r'output/residualMoransI.shp'
gdf.to_file(output_path, driver='ESRI Shapefile')
print(f"Đã lưu kết quả ra {output_path}")

C:\Users\Ducnghia\AppData\Local\Temp\ipykernel_12124\3550080397.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(output_path, driver='ESRI Shapefile')
c:\Users\Ducnghia\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'pop_density' to 'pop_densit'
  ogr_write(


Đã lưu kết quả ra output/residualMoransI.shp
